# PokéRAG: chat por terminal

Este notebook monta un sistema **RAG** sencillo sobre Pokémon:

1. Carga las fichas de Pokémon desde `data/pokemon_data.json`.
2. Convierte cada ficha en un *embedding* para poder buscar por significado.
3. Recupera las fichas más parecidas a la pregunta del usuario.
4. Usa un modelo de lenguaje para responder **solo** con esa información.
5. Permite conversar con el modelo por terminal desde la última celda.

Ejecuta las celdas en orden de arriba a abajo.

## 1. Importar librerías

Cargamos las librerías necesarias para leer los datos, calcular embeddings y generar texto.

In [6]:
import json
import os
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import GenerationConfig, pipeline

os.environ["USE_TF"] = "0"

print("PyTorch versión:", torch.__version__)

PyTorch versión: 2.14.0+cpu


## 2. Cargar los documentos y los modelos

- **Documentos**: fichas de Pokémon generadas por `download_data.py`.
- **Modelo de embeddings**: `all-MiniLM-L6-v2`, convierte texto en vectores numéricos.
- **Modelo generador**: `Qwen2.5-0.5B-Instruct`, redacta la respuesta final.

> Si `data/pokemon_data.json` no existe, ejecuta antes `python download_data.py`.

In [7]:
# 1. Cargar base de datos documental
with open("data/pokemon_data.json", "r", encoding="utf-8") as f:
    documentos = json.load(f)

print(f"Documentos cargados: {len(documentos)}")

# 2. Cargar modelos RAG
modelo_embeddings = SentenceTransformer("all-MiniLM-L6-v2")

generador = pipeline(
    task="text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device=-1  # -1 = CPU
)

print("Modelos cargados con éxito.")

Documentos cargados: 1


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 376.12it/s]


Modelos cargados con éxito.


## 3. Calcular los embeddings de los documentos

Cada documento se transforma en un vector. Después podremos comparar la pregunta del usuario con esos vectores para encontrar las fichas más relevantes.

In [8]:
textos_documentos = [doc["texto"] for doc in documentos]

embeddings_documentos = modelo_embeddings.encode(
    textos_documentos,
    convert_to_numpy=True
)

print("Forma de la matriz de embeddings:", embeddings_documentos.shape)

Forma de la matriz de embeddings: (1, 384)


## 4. Funciones del sistema RAG

- `recuperar_documentos`: busca las `k` fichas más parecidas a la pregunta.
- `construir_contexto`: junta esas fichas en un único texto.
- `generar_respuesta`: pide al modelo que responda usando solo ese contexto.
- `consultar_rag`: une los tres pasos anteriores.

El modelo usa la **plantilla de chat** de Qwen y devuelve únicamente la respuesta nueva, sin repetir el prompt.

In [9]:
def recuperar_documentos(pregunta, k=2):
    """Devuelve las k fichas más parecidas semánticamente a la pregunta."""
    embedding_pregunta = modelo_embeddings.encode(pregunta, convert_to_numpy=True)
    puntuaciones = cosine_similarity([embedding_pregunta], embeddings_documentos)[0]
    indices = np.argsort(puntuaciones)[::-1][:k]

    return [
        {"documento": documentos[indice], "puntuacion": float(puntuaciones[indice])}
        for indice in indices
    ]


def construir_contexto(resultados):
    """Convierte los documentos recuperados en texto para el modelo."""
    partes = []
    for res in resultados:
        doc = res["documento"]
        partes.append(f"[Documento {doc['id']} | Fuente: {doc['fuente']}]\n{doc['texto']}")
    return "\n\n".join(partes)


# Configuración de generación (evita avisos y limita la longitud de la respuesta)
config_generacion = GenerationConfig(
    max_new_tokens=150,
    do_sample=False,
    eos_token_id=generador.tokenizer.eos_token_id,
    pad_token_id=generador.tokenizer.eos_token_id,
)


def generar_respuesta(pregunta, contexto):
    """Genera una respuesta usando únicamente el contexto recuperado."""
    mensajes = [
        {
            "role": "system",
            "content": (
                "Eres un asistente experto en Pokémon. Responde en español y "
                "utiliza únicamente la información del contexto. Si la respuesta "
                "no está en el contexto, di exactamente: 'No encuentro esa "
                "información en los documentos disponibles.' No inventes datos."
            ),
        },
        {
            "role": "user",
            "content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA:\n{pregunta}",
        },
    ]

    prompt = generador.tokenizer.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True,
    )

    salida = generador(
        prompt,
        generation_config=config_generacion,
        return_full_text=False,  # devuelve solo la respuesta, no el prompt
    )
    return salida[0]["generated_text"].strip()


def consultar_rag(pregunta, k=2):
    """Pipeline completo: recuperar contexto y generar la respuesta."""
    resultados = recuperar_documentos(pregunta, k=k)
    contexto = construir_contexto(resultados)
    respuesta = generar_respuesta(pregunta, contexto)

    return {
        "pregunta": pregunta,
        "respuesta": respuesta,
        "documentos_recuperados": resultados,
    }

## 5. Prueba rápida

Comprobamos que el sistema recupera el documento correcto y responde bien antes de abrir el chat.

In [10]:
resultado = consultar_rag("Acabo de capturar un Charmander, ¿a qué nivel evoluciona?")

print("PREGUNTA:", resultado["pregunta"])
print("RESPUESTA:", resultado["respuesta"])
print("DOCUMENTOS:", [d["documento"]["id"] for d in resultado["documentos_recuperados"]])

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


PREGUNTA: Acabo de capturar un Charmander, ¿a qué nivel evoluciona?
RESPUESTA: El Charmander que capturaste se evolucionará a Charmeleon al nivel 16.
DOCUMENTOS: [4]


## 6. Chat por terminal

Ejecuta esta celda y escribe tus preguntas en el cuadro de texto que aparece. Pulsa **Enter** para enviar cada pregunta.

Para terminar la conversación escribe `salir`, `exit` o `q`.

> En Jupyter, la celda se queda ejecutándose mientras dure el chat. Verás las respuestas debajo de la celda.

In [11]:
# Chat interactivo por terminal.
# Escribe una pregunta y pulsa Enter. Para salir: salir, exit o q.

print("PokéRAG listo. Escribe 'salir' para terminar.\n")

while True:
    pregunta = input("Tú: ").strip()

    if pregunta.lower() in {"salir", "exit", "q"}:
        print("¡Hasta luego!")
        break

    if not pregunta:
        continue

    resultado = consultar_rag(pregunta)
    print(f"\nPokéRAG: {resultado['respuesta']}\n")

PokéRAG listo. Escribe 'salir' para terminar.


PokéRAG: Hola! ¿Cómo puedo ayudarte hoy?


PokéRAG: Charmander evoluciona a Charmeleon al nivel 16.


PokéRAG: Charmeleon evoluciona a Charizard al nivel 36.

¡Hasta luego!
